**GloVe Word Embeddings – Without Pretrained Model**



In [ ]:

import numpy as np
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity
with open("glove_corpus.txt", "r") as file:
    sentences = file.readlines()

corpus = [sentence.lower().split() for sentence in sentences]

vocab = sorted(set(word for sentence in corpus for word in sentence))

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

print("Vocabulary Size:", len(vocab))
window_size = 2
co_matrix = defaultdict(float)

for sentence in corpus:
    for i, word in enumerate(sentence):
        word_id = word_to_id[word]

        start = max(0, i - window_size)
        end = min(len(sentence), i + window_size + 1)

        for j in range(start, end):
            if i != j:
                context_word = sentence[j]
                context_id = word_to_id[context_word]

                distance = abs(i - j)
                co_matrix[(word_id, context_id)] += 1 / distance

vocab_size = len(vocab)
embedding_dim = 50

W = np.random.rand(vocab_size, embedding_dim) * 0.01
W_context = np.random.rand(vocab_size, embedding_dim) * 0.01

b = np.zeros(vocab_size)
b_context = np.zeros(vocab_size)


learning_rate = 0.05
epochs = 100
x_max = 100
alpha = 0.75

for epoch in range(epochs):
    cost = 0

    for (i, j), Xij in co_matrix.items():

        weight = (Xij / x_max) ** alpha if Xij < x_max else 1

        prediction = np.dot(W[i], W_context[j]) + b[i] + b_context[j]
        error = prediction - np.log(Xij)
        grad = weight * error

        W[i] -= learning_rate * grad * W_context[j]
        W_context[j] -= learning_rate * grad * W[i]

        b[i] -= learning_rate * grad
        b_context[j] -= learning_rate * grad

        cost += 0.5 * weight * error ** 2

    if epoch % 10 == 0:
        print("Epoch:", epoch, "Cost:", cost)

embeddings = W + W_context
def find_similar_words(word, top_n=5):

    word = word.lower()

    if word not in word_to_id:
        print("Word not found in vocabulary")
        return

    word_vector = embeddings[word_to_id[word]].reshape(1, -1)

    similarities = cosine_similarity(word_vector, embeddings)[0]

    similar_ids = similarities.argsort()[::-1]

    print("\nWords most similar to:", word)

    count = 0
    for idx in similar_ids:
        if vocab[idx] != word:
            print(vocab[idx], ":", similarities[idx])
            count += 1

            if count == top_n:
                break

find_similar_words("king")


Vocabulary Size: 73
Epoch: 0 Cost: 1.9595447627334772
Epoch: 10 Cost: 1.5946193746517185
Epoch: 20 Cost: 1.4700004112330074
Epoch: 30 Cost: 1.4117104202167992
Epoch: 40 Cost: 1.3746716821469793
Epoch: 50 Cost: 1.3463753966638057
Epoch: 60 Cost: 1.3228739252909538
Epoch: 70 Cost: 1.3026359371313203
Epoch: 80 Cost: 1.284898567603377
Epoch: 90 Cost: 1.2691900420604552

Words most similar to: king
the : 0.9610076564974345
is : 0.9184242692849597
rules : 0.9087526960966352
man : 0.9042285807409324
of : 0.9014953032920257


**GloVe Word Embeddings – Using Pretrained Gensim Model**

In [ ]:
!pip install gensim
import gensim.downloader as api
print("Loading GloVe model...")

model = api.load("glove-wiki-gigaword-100")
similar_words = model.most_similar("king", topn=5)
print("\nWords most similar to King:")

for word, similarity in similar_words:
    print(word, ":", similarity)
print("\nSimilarity between King and Queen:")
print(model.similarity("king", "queen"))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 20.3 MB/s  0:00:01
Loading GloVe model...
[==================================================] 100.0% 128.1/128.1MB downloaded

Words most similar to King:
prince : 0.7682328820228577
queen : 0.7507690787315369
son : 0.7020888328552246
brother : 0.6985775232315063
monarch : 0.6977890729904175

Similarity between King and Queen:
0.7507691


**Statistical Language Model Using N-grams**

In [1]:

import re
from collections import defaultdict, Counter

# Step 1: Define the corpus
corpus = [
    "natural language processing is a branch of artificial intelligence.",
    "natural language processing enables computers to understand human language.",
    "machine learning is a branch of artificial intelligence.",
    "deep learning is a part of machine learning.",
    "language models learn patterns from text data.",
    "a language model predicts the next word in a sentence.",
    "natural language processing uses language models.",
    "artificial intelligence includes machine learning and deep learning."
]

# Step 2: Preprocess the corpus
sentences = []

for sentence in corpus:
    sentence = sentence.lower()
    words = re.findall(r'\b\w+\b', sentence)
    sentences.append(words)

# Step 3: Create Bigram and Trigram models
bigram_counts = defaultdict(Counter)
trigram_counts = defaultdict(Counter)

unigram_counts = Counter()

for words in sentences:

    # Count unigrams
    for word in words:
        unigram_counts[word] += 1

    # Count bigrams
    for i in range(len(words) - 1):
        bigram_counts[words[i]][words[i+1]] += 1

    # Count trigrams
    for i in range(len(words) - 2):
        context = (words[i], words[i+1])
        trigram_counts[context][words[i+2]] += 1

# Step 4: Bigram next-word prediction
def predict_bigram(word):

    word = word.lower()

    if word not in bigram_counts:
        print("No prediction available")
        return

    next_words = bigram_counts[word]

    predicted_word = next_words.most_common(1)[0][0]

    print("Input word:", word)
    print("Predicted next word:", predicted_word)


# Step 5: Trigram next-word prediction
def predict_trigram(word1, word2):

    context = (word1.lower(), word2.lower())

    if context not in trigram_counts:
        print("No prediction available")
        return

    next_words = trigram_counts[context]

    predicted_word = next_words.most_common(1)[0][0]

    print("Input words:", context)
    print("Predicted next word:", predicted_word)


# Step 6: Test Bigram model
print("BIGRAM PREDICTION")
predict_bigram("machine")

# Step 7: Test Trigram model
print("\nTRIGRAM PREDICTION")
predict_trigram("machine", "learning")


BIGRAM PREDICTION
Input word: machine
Predicted next word: learning

TRIGRAM PREDICTION
Input words: ('machine', 'learning')
Predicted next word: is
